In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")
OUT_DIR = Path("cleaned_data")
OUT_DIR.mkdir(exist_ok=True)

In [2]:
files = list(DATA_DIR.glob("*.csv"))

dfs = {}

for file in files:
    name = file.stem
    df = pd.read_csv(file)
    dfs[name] = df
    print(name, df.shape)
    print(df.columns.tolist())
    print("-" * 80)

ai_gpu_related_services (52, 10)
['provider', 'service_code', 'service_name', 'service_title', 'category', 'source', 'collection_date', 'category_v2', 'category_v3', 'service_origin']
--------------------------------------------------------------------------------
ai_readiness_index_v1 (3, 12)
['provider', 'ai_service_count', 'ai_type_diversity', 'infra_breadth', 'ai_share_pct', 'avg_hourly_price', 'ai_service_score', 'diversity_score', 'infra_score', 'share_score', 'pricing_score', 'ai_readiness_index']
--------------------------------------------------------------------------------
ai_type_summary (8, 3)
['provider', 'ai_type', 'service_count']
--------------------------------------------------------------------------------
aws_gpu_market_dataset (85, 5)
['Instance Type', 'Instance Family', 'GPU', 'GPU Memory', 'accelerator']
--------------------------------------------------------------------------------
aws_locations (34, 6)
['provider', 'region_code', 'region_name', 'city', 'conti

In [3]:
quality_report = []

for name, df in dfs.items():
    quality_report.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
        "missing_pct": round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2)
    })

quality_df = pd.DataFrame(quality_report)
quality_df.sort_values("missing_pct", ascending=False)

,dataset,rows,columns,duplicate_rows,missing_cells,missing_pct
21,normalized_vm_pricing,97,11,0,164,15.37
9,cloud_locations_dataset,191,6,57,114,9.95
3,aws_gpu_market_dataset,85,5,0,9,2.12
2,ai_type_summary,8,3,0,0,0.00
1,ai_readiness_index_v1,3,12,0,0,0.00
5,azure_gpu_market_dataset,6845,6,109,0,0.00
4,aws_locations,34,6,0,0,0.00
6,azure_gpu_summary,8,4,0,0,0.00
7,azure_locations,114,5,57,0,0.00
8,benchmark_pricing_summary,10,8,0,0,0.00


In [4]:
quality_df.to_csv(OUT_DIR / "data_quality_report.csv", index=False)

In [5]:
azure_locations = dfs["azure_locations"].copy()

azure_locations_clean = (
    azure_locations
    .drop_duplicates(subset=["region_code"])
    .reset_index(drop=True)
)

print("Original Azure rows:", len(azure_locations))
print("Unique Azure regions:", len(azure_locations_clean))

azure_locations_clean.to_csv(OUT_DIR / "azure_locations_clean.csv", index=False)

Original Azure rows: 114
Unique Azure regions: 57


In [6]:
aws_locations = dfs["aws_locations"].copy()
gcp_locations = dfs["gcp_locations"].copy()

region_master = pd.concat(
    [
        aws_locations,
        azure_locations_clean,
        gcp_locations
    ],
    ignore_index=True
)

region_master = region_master.drop_duplicates(
    subset=["provider", "region_code"]
)

region_master.to_csv(OUT_DIR / "master_region_table.csv", index=False)

region_summary_clean = (
    region_master
    .groupby("provider")
    .agg(
        region_count=("region_code", "nunique"),
        country_count=("country", "nunique"),
        continent_count=("continent", "nunique")
    )
    .reset_index()
)

region_summary_clean

,provider,region_count,country_count,continent_count
0,AWS,34,26,11
1,Azure,57,30,0
2,GCP,43,38,7


In [7]:
region_summary_clean.to_csv(OUT_DIR / "region_summary_clean.csv", index=False)

In [8]:
market_share = dfs["cloud_market_share_long"].copy()

market_share["market_share"] = pd.to_numeric(
    market_share["market_share"], errors="coerce"
)

market_share_clean = market_share.dropna()

market_share_clean.to_csv(OUT_DIR / "market_share_clean.csv", index=False)

market_share_clean

,year_quarter,provider,market_share
0,2024 Q4,AWS,30
1,2025 Q1,AWS,32
2,2025 Q2,AWS,30
3,2025 Q3,AWS,29
4,2025 Q4,AWS,28
5,2026 Q1,AWS,28
6,2024 Q4,Azure,21
7,2025 Q1,Azure,21
8,2025 Q2,Azure,20
9,2025 Q3,Azure,20


In [9]:
azure_gpu = dfs["azure_gpu_market_dataset"].copy()

azure_gpu_clean = azure_gpu.copy()
azure_gpu_clean["hourly_price_usd"] = pd.to_numeric(
    azure_gpu_clean["hourly_price_usd"], errors="coerce"
)

azure_gpu_clean = azure_gpu_clean.dropna(subset=["hourly_price_usd"])
azure_gpu_clean = azure_gpu_clean[azure_gpu_clean["hourly_price_usd"] > 0]

azure_gpu_clean.to_csv(OUT_DIR / "azure_gpu_pricing_clean.csv", index=False)

azure_gpu_clean.head()

,provider,region,instance_type,product_name,hourly_price_usd,gpu_type
0,Azure,germanywestcentral,Standard_NC80adis_H100_v5,Virtual Machines NCadsH100v5 Series,3.630000,H100
1,Azure,eastus2,Standard_NC320lds_xl_RTXPRO6000BSE_v6,Virtual Machines NCldsxlRTX6kv6,2.860000,RTX PRO 6000
2,Azure,eastus,Standard_NC320ds_xl_RTXPRO6000BSE_v6,Virtual Machines NCdsxlRTX6kv6 Windows,6.261600,RTX PRO 6000
3,Azure,switzerlandwest,Standard_ND96ams_A100_v4,Virtual Machines NDamsr A100 v4 Series Linux,15.838940,A100
4,Azure,usgovarizona,Standard_NV4as_v4,Virtual Machines NVasv4 Series,0.053777,Other


In [10]:
gcp_gpu = dfs["gcp_accelerator_dataset"].copy()

gcp_gpu_clean = gcp_gpu.copy()
gcp_gpu_clean["hourly_price_usd"] = pd.to_numeric(
    gcp_gpu_clean["hourly_price_usd"], errors="coerce"
)

gcp_gpu_clean = gcp_gpu_clean.dropna(subset=["hourly_price_usd"])
gcp_gpu_clean = gcp_gpu_clean[gcp_gpu_clean["hourly_price_usd"] > 0]

gcp_gpu_clean.to_csv(OUT_DIR / "gcp_gpu_pricing_clean.csv", index=False)

gcp_gpu_clean.head()

,provider,sku_id,description,resource_family,resource_group,usage_type,regions,hourly_price_usd,currency,source,accelerator
0,GCP,0008-F633-76AA,Nvidia L4 GPU attached to Spot Preemptible VMs...,Compute,GPU,Preemptible,asia-east2,0.21210,USD,GCP Cloud Billing Catalog API,L4
1,GCP,0032-6F6D-C48E,Nvidia L4 GPU attached to Spot Preemptible VMs...,Compute,GPU,Preemptible,northamerica-northeast1,0.30580,USD,GCP Cloud Billing Catalog API,L4
2,GCP,003E-D940-4BC0,Nvidia Tesla P100 GPU running in Seoul,Compute,GPU,OnDemand,asia-northeast3,1.60000,USD,GCP Cloud Billing Catalog API,P100
3,GCP,0050-986A-2850,Commitment v1: H200 141GB GPU running in Nethe...,Compute,GPU,Commit1Yr,europe-west4,7.07453,USD,GCP Cloud Billing Catalog API,H200
4,GCP,005F-DC8C-CB19,Nvidia H100 80GB Mega GPU running in Sydney,Compute,GPU,OnDemand,australia-southeast1,12.93034,USD,GCP Cloud Billing Catalog API,H100


In [11]:
aws_gpu = dfs["aws_gpu_market_dataset"].copy()

aws_gpu_clean = aws_gpu.drop_duplicates()

aws_gpu_clean.to_csv(OUT_DIR / "aws_gpu_availability_clean.csv", index=False)

aws_gpu_clean.head()

,Instance Type,Instance Family,GPU,GPU Memory,accelerator
0,g7e.24xlarge,GPU instance,4.0,384 GB,Blackwell
1,g6.48xlarge,GPU instance,8.0,192 GB,L4
2,inf1.xlarge,Machine Learning ASIC Instances,1.0,NaN,Inferentia
3,inf1.2xlarge,Machine Learning ASIC Instances,1.0,NaN,Inferentia
4,gr6.8xlarge,GPU instance,1.0,24 GB,L4


In [12]:
ai_services = dfs["native_ai_services_enriched"].copy()

ai_services_clean = ai_services.drop_duplicates(
    subset=["provider", "service_code", "service_name"]
)

ai_service_summary_clean = (
    ai_services_clean
    .groupby("provider")
    .agg(
        ai_service_count=("service_name", "nunique"),
        ai_category_count=("category_v3", "nunique")
    )
    .reset_index()
)

ai_services_clean.to_csv(OUT_DIR / "ai_services_clean.csv", index=False)
ai_service_summary_clean.to_csv(OUT_DIR / "ai_service_summary_clean.csv", index=False)

ai_service_summary_clean

,provider,ai_service_count,ai_category_count
0,AWS,6,1
1,Azure,4,1
2,GCP,7,1


In [13]:
native_services = dfs["native_cloud_service_catalog"].copy()

native_services_clean = native_services.drop_duplicates(
    subset=["provider", "service_code", "service_name"]
)

native_service_summary_clean = (
    native_services_clean
    .groupby("provider")
    .agg(
        native_service_count=("service_name", "nunique"),
        category_count=("category_v3", "nunique")
    )
    .reset_index()
)

native_services_clean.to_csv(OUT_DIR / "native_services_clean.csv", index=False)
native_service_summary_clean.to_csv(OUT_DIR / "native_service_summary_clean.csv", index=False)

native_service_summary_clean

,provider,native_service_count,category_count
0,AWS,69,9
1,Azure,32,9
2,GCP,351,9


In [14]:
kpi_table = (
    region_summary_clean
    .merge(ai_service_summary_clean, on="provider", how="left")
    .merge(native_service_summary_clean, on="provider", how="left")
)

latest_market_share = (
    market_share_clean
    .sort_values("year_quarter")
    .groupby("provider")
    .tail(1)
    [["provider", "year_quarter", "market_share"]]
)

kpi_table = kpi_table.merge(latest_market_share, on="provider", how="left")

kpi_table.to_csv(OUT_DIR / "executive_kpi_table.csv", index=False)

kpi_table

,provider,region_count,country_count,continent_count,ai_service_count,ai_category_count,native_service_count,category_count,year_quarter,market_share
0,AWS,34,26,11,6,1,69,9,2026 Q1,28
1,Azure,57,30,0,4,1,32,9,2026 Q1,21
2,GCP,43,38,7,7,1,351,9,2026 Q1,15


In [15]:
for file in OUT_DIR.glob("*.csv"):
    print(file.name)

ai_services_clean.csv
ai_service_summary_clean.csv
aws_gpu_availability_clean.csv
azure_gpu_pricing_clean.csv
azure_locations_clean.csv
data_quality_report.csv
executive_kpi_table.csv
gcp_gpu_pricing_clean.csv
market_share_clean.csv
master_region_table.csv
native_services_clean.csv
native_service_summary_clean.csv
region_summary_clean.csv


In [16]:
import pandas as pd
import os

for file in os.listdir():
    if file.endswith(".csv"):
        df = pd.read_csv(file)

        print("="*80)
        print(file)
        print("Rows:", len(df))
        print("Columns:", len(df.columns))
        print(df.columns.tolist())
        print()

ai_gpu_related_services.csv
Rows: 52
Columns: 10
['provider', 'service_code', 'service_name', 'service_title', 'category', 'source', 'collection_date', 'category_v2', 'category_v3', 'service_origin']

ai_readiness_index_v1.csv
Rows: 3
Columns: 12
['provider', 'ai_service_count', 'ai_type_diversity', 'infra_breadth', 'ai_share_pct', 'avg_hourly_price', 'ai_service_score', 'diversity_score', 'infra_score', 'share_score', 'pricing_score', 'ai_readiness_index']

ai_type_summary.csv
Rows: 8
Columns: 3
['provider', 'ai_type', 'service_count']

aws_gpu_market_dataset.csv
Rows: 85
Columns: 5
['Instance Type', 'Instance Family', 'GPU', 'GPU Memory', 'accelerator']

aws_locations.csv
Rows: 34
Columns: 6
['provider', 'region_code', 'region_name', 'city', 'continent', 'country']

azure_gpu_market_dataset.csv
Rows: 6845
Columns: 6
['provider', 'region', 'instance_type', 'product_name', 'hourly_price_usd', 'gpu_type']

azure_gpu_summary.csv
Rows: 8
Columns: 4
['gpu_type', 'vm_count', 'unique_vm', 'a

In [17]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")
OUT_DIR = Path("cleaned_data")
OUT_DIR.mkdir(exist_ok=True)

dfs = {}

for file in DATA_DIR.glob("*.csv"):
    df = pd.read_csv(file)
    dfs[file.stem] = df
    print(file.name, df.shape)

ai_gpu_related_services.csv (52, 10)
ai_readiness_index_v1.csv (3, 12)
ai_type_summary.csv (8, 3)
aws_gpu_market_dataset.csv (85, 5)
aws_locations.csv (34, 6)
azure_gpu_market_dataset.csv (6845, 6)
azure_gpu_summary.csv (8, 4)
azure_locations.csv (114, 5)
benchmark_pricing_summary.csv (10, 8)
cloud_locations_dataset.csv (191, 6)
cloud_market_share_long.csv (18, 3)
cloud_market_share_wide.csv (6, 4)
country_coverage.csv (3, 2)
gcp_accelerator_dataset.csv (2394, 11)
gcp_accelerator_summary.csv (14, 4)
gcp_locations.csv (43, 6)
global_region_detail.csv (159, 2)
global_region_summary.csv (3, 2)
native_ai_services_enriched.csv (17, 11)
native_cloud_service_catalog.csv (460, 9)
native_service_category_summary.csv (30, 3)
normalized_vm_pricing.csv (97, 11)
other_marketplace_service_catalog.csv (966, 9)
provider_ai_service_share.csv (3, 4)
region_summary.csv (3, 2)
service_category_pivot_summary.csv (9, 4)
service_category_summary.csv (26, 3)


In [18]:
quality_report = []

for name, df in dfs.items():
    quality_report.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
        "missing_pct": round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2)
    })

quality_df = pd.DataFrame(quality_report)
quality_df = quality_df.sort_values("missing_pct", ascending=False)

quality_df.to_csv(OUT_DIR / "data_quality_report.csv", index=False)
quality_df

,dataset,rows,columns,duplicate_rows,missing_cells,missing_pct
21,normalized_vm_pricing,97,11,0,164,15.37
9,cloud_locations_dataset,191,6,57,114,9.95
3,aws_gpu_market_dataset,85,5,0,9,2.12
2,ai_type_summary,8,3,0,0,0.00
1,ai_readiness_index_v1,3,12,0,0,0.00
5,azure_gpu_market_dataset,6845,6,109,0,0.00
4,aws_locations,34,6,0,0,0.00
6,azure_gpu_summary,8,4,0,0,0.00
7,azure_locations,114,5,57,0,0.00
8,benchmark_pricing_summary,10,8,0,0,0.00


In [19]:
aws_regions = dfs["aws_locations"].copy()
azure_regions = dfs["azure_locations"].copy()
gcp_regions = dfs["gcp_locations"].copy()

# Azure 沒有 continent，先補上 Unknown
azure_regions["continent"] = "Unknown"

# 統一欄位順序
region_cols = ["provider", "region_code", "region_name", "city", "country", "continent"]

aws_regions = aws_regions[region_cols]
azure_regions = azure_regions[region_cols]
gcp_regions = gcp_regions[region_cols]

# 去重複
aws_regions_clean = aws_regions.drop_duplicates(subset=["provider", "region_code"])
azure_regions_clean = azure_regions.drop_duplicates(subset=["provider", "region_code"])
gcp_regions_clean = gcp_regions.drop_duplicates(subset=["provider", "region_code"])

master_region_table = pd.concat(
    [aws_regions_clean, azure_regions_clean, gcp_regions_clean],
    ignore_index=True
)

master_region_table = master_region_table.drop_duplicates(
    subset=["provider", "region_code"]
)

region_summary_clean = (
    master_region_table
    .groupby("provider")
    .agg(
        region_count=("region_code", "nunique"),
        country_count=("country", "nunique"),
        continent_count=("continent", "nunique")
    )
    .reset_index()
)

master_region_table.to_csv(OUT_DIR / "master_region_table.csv", index=False)
region_summary_clean.to_csv(OUT_DIR / "region_summary_clean.csv", index=False)

region_summary_clean

,provider,region_count,country_count,continent_count
0,AWS,34,26,11
1,Azure,57,30,1
2,GCP,43,38,7


In [20]:
original_region_summary = dfs["region_summary"].copy()
global_region_summary = dfs["global_region_summary"].copy()

comparison_region = (
    region_summary_clean[["provider", "region_count"]]
    .rename(columns={"region_count": "recalculated_region_count"})
    .merge(original_region_summary.rename(columns={"region_count": "region_summary_count"}), on="provider", how="left")
    .merge(global_region_summary.rename(columns={"region_count": "global_region_summary_count"}), on="provider", how="left")
)

comparison_region.to_csv(OUT_DIR / "region_count_comparison.csv", index=False)
comparison_region

,provider,recalculated_region_count,region_summary_count,global_region_summary_count
0,AWS,34,34,42
1,Azure,57,114,77
2,GCP,43,43,40


In [21]:
market_share = dfs["cloud_market_share_long"].copy()

market_share["market_share"] = pd.to_numeric(market_share["market_share"], errors="coerce")
market_share_clean = market_share.dropna(subset=["market_share"])

market_share_clean.to_csv(OUT_DIR / "market_share_clean.csv", index=False)

market_share_clean

,year_quarter,provider,market_share
0,2024 Q4,AWS,30
1,2025 Q1,AWS,32
2,2025 Q2,AWS,30
3,2025 Q3,AWS,29
4,2025 Q4,AWS,28
5,2026 Q1,AWS,28
6,2024 Q4,Azure,21
7,2025 Q1,Azure,21
8,2025 Q2,Azure,20
9,2025 Q3,Azure,20


In [22]:
azure_gpu = dfs["azure_gpu_market_dataset"].copy()

azure_gpu["hourly_price_usd"] = pd.to_numeric(
    azure_gpu["hourly_price_usd"], errors="coerce"
)

azure_gpu_clean = azure_gpu.dropna(subset=["hourly_price_usd"])
azure_gpu_clean = azure_gpu_clean[azure_gpu_clean["hourly_price_usd"] > 0]
azure_gpu_clean = azure_gpu_clean.drop_duplicates()

azure_gpu_clean.to_csv(OUT_DIR / "azure_gpu_pricing_clean.csv", index=False)

azure_gpu_clean.head()

,provider,region,instance_type,product_name,hourly_price_usd,gpu_type
0,Azure,germanywestcentral,Standard_NC80adis_H100_v5,Virtual Machines NCadsH100v5 Series,3.630000,H100
1,Azure,eastus2,Standard_NC320lds_xl_RTXPRO6000BSE_v6,Virtual Machines NCldsxlRTX6kv6,2.860000,RTX PRO 6000
2,Azure,eastus,Standard_NC320ds_xl_RTXPRO6000BSE_v6,Virtual Machines NCdsxlRTX6kv6 Windows,6.261600,RTX PRO 6000
3,Azure,switzerlandwest,Standard_ND96ams_A100_v4,Virtual Machines NDamsr A100 v4 Series Linux,15.838940,A100
4,Azure,usgovarizona,Standard_NV4as_v4,Virtual Machines NVasv4 Series,0.053777,Other


In [23]:
gcp_gpu = dfs["gcp_accelerator_dataset"].copy()

gcp_gpu["hourly_price_usd"] = pd.to_numeric(
    gcp_gpu["hourly_price_usd"], errors="coerce"
)

gcp_gpu_clean = gcp_gpu.dropna(subset=["hourly_price_usd"])
gcp_gpu_clean = gcp_gpu_clean[gcp_gpu_clean["hourly_price_usd"] > 0]
gcp_gpu_clean = gcp_gpu_clean.drop_duplicates()

gcp_gpu_clean.to_csv(OUT_DIR / "gcp_gpu_pricing_clean.csv", index=False)

gcp_gpu_clean.head()

,provider,sku_id,description,resource_family,resource_group,usage_type,regions,hourly_price_usd,currency,source,accelerator
0,GCP,0008-F633-76AA,Nvidia L4 GPU attached to Spot Preemptible VMs...,Compute,GPU,Preemptible,asia-east2,0.21210,USD,GCP Cloud Billing Catalog API,L4
1,GCP,0032-6F6D-C48E,Nvidia L4 GPU attached to Spot Preemptible VMs...,Compute,GPU,Preemptible,northamerica-northeast1,0.30580,USD,GCP Cloud Billing Catalog API,L4
2,GCP,003E-D940-4BC0,Nvidia Tesla P100 GPU running in Seoul,Compute,GPU,OnDemand,asia-northeast3,1.60000,USD,GCP Cloud Billing Catalog API,P100
3,GCP,0050-986A-2850,Commitment v1: H200 141GB GPU running in Nethe...,Compute,GPU,Commit1Yr,europe-west4,7.07453,USD,GCP Cloud Billing Catalog API,H200
4,GCP,005F-DC8C-CB19,Nvidia H100 80GB Mega GPU running in Sydney,Compute,GPU,OnDemand,australia-southeast1,12.93034,USD,GCP Cloud Billing Catalog API,H100


In [24]:
aws_gpu = dfs["aws_gpu_market_dataset"].copy()

aws_gpu_clean = aws_gpu.drop_duplicates()

aws_gpu_clean.to_csv(OUT_DIR / "aws_gpu_availability_clean.csv", index=False)

aws_gpu_clean.head()

,Instance Type,Instance Family,GPU,GPU Memory,accelerator
0,g7e.24xlarge,GPU instance,4.0,384 GB,Blackwell
1,g6.48xlarge,GPU instance,8.0,192 GB,L4
2,inf1.xlarge,Machine Learning ASIC Instances,1.0,NaN,Inferentia
3,inf1.2xlarge,Machine Learning ASIC Instances,1.0,NaN,Inferentia
4,gr6.8xlarge,GPU instance,1.0,24 GB,L4


In [25]:
azure_gpu_summary_clean = (
    azure_gpu_clean
    .groupby(["provider", "gpu_type"])
    .agg(
        record_count=("instance_type", "count"),
        unique_instance_count=("instance_type", "nunique"),
        avg_hourly_price=("hourly_price_usd", "mean"),
        median_hourly_price=("hourly_price_usd", "median")
    )
    .reset_index()
)

gcp_gpu_summary_clean = (
    gcp_gpu_clean
    .groupby(["provider", "accelerator"])
    .agg(
        record_count=("sku_id", "count"),
        avg_hourly_price=("hourly_price_usd", "mean"),
        median_hourly_price=("hourly_price_usd", "median")
    )
    .reset_index()
    .rename(columns={"accelerator": "gpu_type"})
)

aws_gpu_summary_clean = (
    aws_gpu_clean
    .groupby("accelerator")
    .agg(
        instance_count=("Instance Type", "nunique"),
        family_count=("Instance Family", "nunique")
    )
    .reset_index()
    .rename(columns={"accelerator": "gpu_type"})
)

aws_gpu_summary_clean["provider"] = "AWS"

gpu_summary_clean = pd.concat(
    [
        azure_gpu_summary_clean[["provider", "gpu_type", "record_count", "unique_instance_count", "avg_hourly_price", "median_hourly_price"]],
        gcp_gpu_summary_clean.assign(unique_instance_count=None)[["provider", "gpu_type", "record_count", "unique_instance_count", "avg_hourly_price", "median_hourly_price"]]
    ],
    ignore_index=True
)

gpu_summary_clean.to_csv(OUT_DIR / "gpu_pricing_summary_clean.csv", index=False)
aws_gpu_summary_clean.to_csv(OUT_DIR / "aws_gpu_summary_clean.csv", index=False)

gpu_summary_clean

,provider,gpu_type,record_count,unique_instance_count,avg_hourly_price,median_hourly_price
0,Azure,A10,1338,9,2.248561,1.174500
1,Azure,A100,864,6,14.376439,9.503000
2,Azure,GB200,42,2,137.002181,135.688000
3,Azure,H100,714,9,39.550567,20.645364
4,Azure,H200,78,2,97.173141,106.088000
5,Azure,MI300X,204,2,32.133292,20.083000
6,Azure,Other,3180,45,2.622457,1.081569
7,Azure,RTX PRO 6000,316,12,6.789159,3.003200
8,GCP,A100,323,None,2.723361,2.388300
9,GCP,Blackwell (B200),136,None,8.057794,8.055000


In [26]:
ai_services = dfs["native_ai_services_enriched"].copy()

ai_services_clean = ai_services.drop_duplicates(
    subset=["provider", "service_code", "service_name"]
)

ai_service_summary_clean = (
    ai_services_clean
    .groupby("provider")
    .agg(
        ai_service_count=("service_name", "nunique"),
        ai_type_count=("ai_type", "nunique"),
        ai_category_count=("category_v3", "nunique")
    )
    .reset_index()
)

ai_services_clean.to_csv(OUT_DIR / "ai_services_clean.csv", index=False)
ai_service_summary_clean.to_csv(OUT_DIR / "ai_service_summary_clean.csv", index=False)

ai_service_summary_clean

,provider,ai_service_count,ai_type_count,ai_category_count
0,AWS,6,3,1
1,Azure,4,3,1
2,GCP,7,2,1


In [27]:
native_services = dfs["native_cloud_service_catalog"].copy()

native_services_clean = native_services.drop_duplicates(
    subset=["provider", "service_code", "service_name"]
)

native_service_summary_clean = (
    native_services_clean
    .groupby("provider")
    .agg(
        native_service_count=("service_name", "nunique"),
        category_count=("category_v3", "nunique")
    )
    .reset_index()
)

native_services_clean.to_csv(OUT_DIR / "native_services_clean.csv", index=False)
native_service_summary_clean.to_csv(OUT_DIR / "native_service_summary_clean.csv", index=False)

native_service_summary_clean

,provider,native_service_count,category_count
0,AWS,69,9
1,Azure,32,9
2,GCP,351,9


In [28]:
latest_market_share = (
    market_share_clean
    .sort_values("year_quarter")
    .groupby("provider")
    .tail(1)
    [["provider", "year_quarter", "market_share"]]
)

executive_kpi_table = (
    region_summary_clean
    .merge(ai_service_summary_clean, on="provider", how="left")
    .merge(native_service_summary_clean, on="provider", how="left")
    .merge(latest_market_share, on="provider", how="left")
)

executive_kpi_table.to_csv(OUT_DIR / "executive_kpi_table.csv", index=False)

executive_kpi_table

,provider,region_count,country_count,continent_count,ai_service_count,ai_type_count,ai_category_count,native_service_count,category_count,year_quarter,market_share
0,AWS,34,26,11,6,3,1,69,9,2026 Q1,28
1,Azure,57,30,1,4,3,1,32,9,2026 Q1,21
2,GCP,43,38,7,7,2,1,351,9,2026 Q1,15


In [29]:
for file in OUT_DIR.glob("*.csv"):
    print(file.name)

ai_services_clean.csv
ai_service_summary_clean.csv
aws_gpu_availability_clean.csv
aws_gpu_summary_clean.csv
azure_gpu_pricing_clean.csv
azure_locations_clean.csv
data_quality_report.csv
executive_kpi_table.csv
gcp_gpu_pricing_clean.csv
gpu_pricing_summary_clean.csv
market_share_clean.csv
master_region_table.csv
native_services_clean.csv
native_service_summary_clean.csv
region_count_comparison.csv
region_summary_clean.csv


In [31]:
OUT_DIR = Path("cleaned_data")
OUT_DIR.mkdir(exist_ok=True)

In [33]:
import os

print(os.listdir("cleaned_data"))

['ai_services_clean.csv', 'ai_service_summary_clean.csv', 'aws_gpu_availability_clean.csv', 'aws_gpu_summary_clean.csv', 'azure_gpu_pricing_clean.csv', 'azure_locations_clean.csv', 'data_quality_report.csv', 'executive_kpi_table.csv', 'gcp_gpu_pricing_clean.csv', 'gpu_pricing_summary_clean.csv', 'market_share_clean.csv', 'master_region_table.csv', 'native_services_clean.csv', 'native_service_summary_clean.csv', 'region_count_comparison.csv', 'region_summary_clean.csv']


In [34]:
from pathlib import Path

OUT_DIR = Path("cleaned_data")

print(OUT_DIR.absolute())

C:\Users\user\Cloud Project\cleaned_data


In [35]:
print(os.getcwd())
print(os.listdir("cleaned_data"))

C:\Users\user\Cloud Project
['ai_services_clean.csv', 'ai_service_summary_clean.csv', 'aws_gpu_availability_clean.csv', 'aws_gpu_summary_clean.csv', 'azure_gpu_pricing_clean.csv', 'azure_locations_clean.csv', 'data_quality_report.csv', 'executive_kpi_table.csv', 'gcp_gpu_pricing_clean.csv', 'gpu_pricing_summary_clean.csv', 'market_share_clean.csv', 'master_region_table.csv', 'native_services_clean.csv', 'native_service_summary_clean.csv', 'region_count_comparison.csv', 'region_summary_clean.csv']


In [37]:
import pandas as pd

azure = pd.read_csv("cleaned_data/azure_locations_clean.csv")
azure["country"].drop_duplicates().sort_values().tolist()

['Asia Pacific',
 'Australia',
 'Austria',
 'Belgium',
 'Brazil',
 'Canada',
 'Chile',
 'Denmark',
 'Europe',
 'France',
 'Germany',
 'India',
 'Indonesia',
 'Israel',
 'Italy',
 'Japan',
 'Korea',
 'Malaysia',
 'Mexico',
 'New Zealand',
 'Norway',
 'Poland',
 'Qatar',
 'South Africa',
 'Spain',
 'Sweden',
 'Switzerland',
 'UAE',
 'United Kingdom',
 'United States']

In [38]:
continent_map = {
    "United States":"North America",
    "Canada":"North America",
    "Mexico":"North America",

    "Brazil":"South America",
    "Chile":"South America",

    "United Kingdom":"Europe",
    "France":"Europe",
    "Germany":"Europe",
    "Austria":"Europe",
    "Belgium":"Europe",
    "Denmark":"Europe",
    "Italy":"Europe",
    "Norway":"Europe",
    "Poland":"Europe",
    "Spain":"Europe",
    "Sweden":"Europe",
    "Switzerland":"Europe",
    "Europe":"Europe",

    "Japan":"Asia",
    "Korea":"Asia",
    "India":"Asia",
    "Indonesia":"Asia",
    "Malaysia":"Asia",
    "Israel":"Asia",
    "Qatar":"Asia",
    "UAE":"Asia",
    "Asia Pacific":"Asia",

    "Australia":"Oceania",
    "New Zealand":"Oceania",

    "South Africa":"Africa"
}

In [39]:
import pandas as pd

azure = pd.read_csv("cleaned_data/azure_locations_clean.csv")

continent_map = {
    "United States":"North America",
    "Canada":"North America",
    "Mexico":"North America",
    "Brazil":"South America",
    "Chile":"South America",
    "United Kingdom":"Europe",
    "France":"Europe",
    "Germany":"Europe",
    "Austria":"Europe",
    "Belgium":"Europe",
    "Denmark":"Europe",
    "Italy":"Europe",
    "Norway":"Europe",
    "Poland":"Europe",
    "Spain":"Europe",
    "Sweden":"Europe",
    "Switzerland":"Europe",
    "Europe":"Europe",
    "Japan":"Asia",
    "Korea":"Asia",
    "India":"Asia",
    "Indonesia":"Asia",
    "Malaysia":"Asia",
    "Israel":"Asia",
    "Qatar":"Asia",
    "UAE":"Asia",
    "Asia Pacific":"Asia",
    "Australia":"Oceania",
    "New Zealand":"Oceania",
    "South Africa":"Africa"
}

azure["continent"] = azure["country"].map(continent_map)

azure.to_csv(
    "cleaned_data/azure_locations_clean.csv",
    index=False
)

In [40]:
master_region_table
region_summary_clean
executive_kpi_table

,provider,region_count,country_count,continent_count,ai_service_count,ai_type_count,ai_category_count,native_service_count,category_count,year_quarter,market_share
0,AWS,34,26,11,6,3,1,69,9,2026 Q1,28
1,Azure,57,30,1,4,3,1,32,9,2026 Q1,21
2,GCP,43,38,7,7,2,1,351,9,2026 Q1,15


In [43]:
native = pd.read_csv("native_cloud_service_catalog.csv")

native.groupby("provider").size()

provider
AWS       69
Azure     32
GCP      359
dtype: int64

In [44]:
native.groupby("provider")["category"].nunique()

provider
AWS      11
Azure     8
GCP      11
Name: category, dtype: int64

In [45]:
native.groupby("provider")["service_name"].nunique()

provider
AWS       69
Azure     32
GCP      351
Name: service_name, dtype: int64

In [46]:
native = pd.read_csv("native_cloud_service_catalog.csv")

native.groupby("provider")["category"].value_counts()

provider  category               
AWS       AI / Machine Learning      14
          Analytics                   8
          Compute                     8
          Security                    7
          Storage                     6
          Billing / Support           5
          DevOps                      5
          Messaging / Integration     5
          Networking                  5
          Database                    3
          Management                  3
Azure     Networking                  6
          Storage                     5
          AI / Machine Learning       4
          Compute                     4
          Database                    4
          Management                  4
          Security                    3
          Analytics                   2
GCP       AI / Machine Learning      85
          Database                   47
          Networking                 41
          Security                   40
          Compute                    37
      

In [47]:
native[native["provider"]=="GCP"][
    ["service_name","category"]
].head(50)

,service_name,category
101,OpenLogic CentOS 7.8 (v20200922) - Security Ha...,Security
102,Commvault Backup & Recovery BYOL,Storage
103,MayaData Cloud File Gateway,Networking
104,Cloud Text-to-Speech API,AI / Machine Learning
105,F5 Networks F5 Per-App VE Advanced WAF (PAYG...,Networking
106,Aiven for Redis,Database
107,Moodle: The Ultimate Learning Management System,Management
108,VMware Tanzu Greenplum BYOL,Compute
109,DataStax API (Dev),Billing / Support
110,PostgreSQL 9.6,Database
